# BentoML: Packaging and Serving ML Models

## What Is BentoML?

Imagine you make an amazing cake (your ML model). To ship it to customers, you need proper packaging — a box, instructions, freshness label, and delivery label.  
**BentoML** is that packaging system for ML models.

A **Bento** (Japanese lunch box) in BentoML is a self-contained package containing:
- Your model weights
- The service code (how to run inference)
- All dependencies (requirements.txt)
- API schema

From a Bento, you can deploy to: REST API, Docker container, Kubernetes, AWS Lambda, etc.

## Resources

- **Docs**: [https://docs.bentoml.com/](https://docs.bentoml.com/)
- **GitHub**: [https://github.com/bentoml/BentoML](https://github.com/bentoml/BentoML)
- **YouTube — BentoML tutorial**: [https://www.youtube.com/watch?v=i4pYQzA0o3I](https://www.youtube.com/watch?v=i4pYQzA0o3I)

## Installation

```bash
pip install bentoml
pip install bentoml[io-image]  # for image I/O support
```

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

try:
    import bentoml
    BENTOML_AVAILABLE = True
    print(f"BentoML version: {bentoml.__version__}")
except ImportError:
    BENTOML_AVAILABLE = False
    print("BentoML not installed — simulated output shown. Install: pip install bentoml")

# Train a pipeline (scaler + model together)
np.random.seed(42)
X, y = make_classification(n_samples=2000, n_features=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])
pipeline.fit(X_train, y_train)
print(f"Pipeline trained: {pipeline.score(X_test, y_test):.3f} accuracy")

## Core Concept 1: Saving Models to BentoML Model Store

BentoML has a **Model Store** — a local registry for your models.  
Each saved model gets a unique tag (name:version) for retrieval.

In [ ]:
if BENTOML_AVAILABLE:
    # Save sklearn pipeline to BentoML model store
    saved_model = bentoml.sklearn.save_model(
        name="churn_classifier",           # model name
        model=pipeline,                     # the sklearn object
        signatures={
            "predict": {"batchable": True, "batch_dim": 0},
            "predict_proba": {"batchable": True, "batch_dim": 0},
        },
        metadata={
            "accuracy": float(pipeline.score(X_test, y_test)),
            "training_samples": len(X_train),
            "features": [f"feature_{i}" for i in range(10)],
        },
        labels={"team": "ml-team", "stage": "production"}
    )

    print(f"Model saved:")
    print(f"  Tag:     {saved_model.tag}")
    print(f"  Name:    {saved_model.tag.name}")
    print(f"  Version: {saved_model.tag.version}")
    print(f"  Path:    {saved_model.path}")

    # List models in store
    print("\nAll models in BentoML store:")
    for m in bentoml.models.list():
        print(f"  {m.tag}  (saved {m.creation_time.strftime('%Y-%m-%d %H:%M')})")

    # Load model back
    loaded = bentoml.sklearn.load_model("churn_classifier:latest")
    test_preds = loaded.predict(X_test[:3])
    print(f"\nLoaded model predictions: {test_preds}")

else:
    print("BentoML save_model (simulated):")
    print()
    print("  saved = bentoml.sklearn.save_model(")
    print("      name='churn_classifier',")
    print("      model=pipeline,")
    print("      signatures={'predict': {'batchable': True}},")
    print("      metadata={'accuracy': 0.893}")
    print("  )")
    print()
    print("  Saved:")
    print("    Tag:     churn_classifier:a2b3c4d5")
    print("    Path:    ~/bentoml/models/churn_classifier/a2b3c4d5")
    print()
    print("  $ bentoml models list")
    print("  Tag                              Size    Creation Time")
    print("  churn_classifier:a2b3c4d5        2.3 MB  2024-01-15 10:30")

## Core Concept 2: Defining a BentoML Service

A **Service** defines:
- Which model to use
- What HTTP endpoints to expose (`@bentoml.api`)
- Input/output types (numpy arrays, DataFrames, images, text, JSON)

Save this as a file (e.g., `service.py`) and run with `bentoml serve service:svc`

In [ ]:
SERVICE_CODE = '''
# service.py — save this file and run: bentoml serve service:svc
import bentoml
import numpy as np
from bentoml.io import NumpyNdarray, JSON
from pydantic import BaseModel
from typing import List

# Load model runner — handles batching and concurrency automatically
model_runner = bentoml.sklearn.get("churn_classifier:latest").to_runner()

# Create service
svc = bentoml.Service(
    name="churn_prediction_service",
    runners=[model_runner]
)

# Define input/output schemas
class PredictionInput(BaseModel):
    features: List[float]  # 10 features

class PredictionOutput(BaseModel):
    prediction: int
    confidence: float
    probabilities: List[float]

@svc.api(input=JSON(pydantic_model=PredictionInput),
         output=JSON(pydantic_model=PredictionOutput))
async def predict(inp: PredictionInput) -> PredictionOutput:
    """Single prediction."""
    features = np.array(inp.features).reshape(1, -1)
    prediction = await model_runner.predict.async_run(features)
    proba = await model_runner.predict_proba.async_run(features)
    return PredictionOutput(
        prediction=int(prediction[0]),
        confidence=float(max(proba[0])),
        probabilities=proba[0].tolist()
    )

@svc.api(input=NumpyNdarray(shape=(-1, 10), dtype=np.float32),
         output=NumpyNdarray(dtype=np.int64))
async def predict_batch(features: np.ndarray) -> np.ndarray:
    """Batch prediction — BentoML handles adaptive batching automatically."""
    return await model_runner.predict.async_run(features)
'''

print("BentoML service definition (service.py):")
print(SERVICE_CODE)

print("Commands to run:")
commands = [
    ("bentoml serve service:svc",           "Start HTTP server at localhost:3000"),
    ("bentoml serve service:svc --reload",  "Auto-reload on file changes (development)"),
    ("bentoml build",                        "Build a Bento (deployable package)"),
    ("bentoml containerize churn_prediction_service:latest",  "Create Docker image"),
    ("docker run -p 3000:3000 churn_prediction_service:latest", "Run Docker container"),
]
for cmd, desc in commands:
    print(f"  $ {cmd}")
    print(f"    → {desc}")

## Core Concept 3: Adaptive Batching

BentoML's **adaptive batching** automatically groups concurrent requests into batches.  
If 100 users send predictions simultaneously, BentoML batches them and calls the model once — 
dramatically improving throughput without changing client code.

In [ ]:
print("Adaptive Batching configuration:")
print()
batching_config = {
    "runner": {
        "batching": {
            "enabled": True,
            "max_batch_size": 100,      # max items per batch
            "max_latency_ms": 5         # wait at most 5ms for more items
        }
    }
}

print("In bentofile.yaml or service config:")
print("  batching:")
print("    enabled: true")
print("    max_batch_size: 100   # group up to 100 requests")
print("    max_latency_ms: 5     # wait max 5ms for a full batch")
print()
print("Without batching: 100 users → 100 model.predict() calls")
print("With batching:    100 users → 1 model.predict(batch_of_100) call")
print()

# Show throughput improvement
n_requests = 100
single_latency_ms = 10    # 10ms per single prediction
batch_latency_ms  = 50    # 50ms for 100 predictions (GPU is efficient)

throughput_single  = 1000 / single_latency_ms   # requests/second (single worker)
throughput_batched = n_requests / (batch_latency_ms / 1000)  # for 100 concurrent requests

print(f"Single prediction:   {single_latency_ms}ms each")
print(f"Batch of 100:        {batch_latency_ms}ms total → {batch_latency_ms/n_requests:.1f}ms per request")
print(f"Throughput (single): {throughput_single:.0f} req/s")
print(f"Throughput (batch):  {throughput_batched:.0f} req/s  ({throughput_batched/throughput_single:.0f}× faster!)")

## Common Pitfalls

| Pitfall | Symptom | Fix |
|---------|---------|-----|
| Calling `load_model` in api method | Slow cold start | Load at module level, not inside each request |
| Not using `to_runner()` | No adaptive batching | Always use Runner for models in services |
| Mismatch: save sklearn, use pytorch | `TypeError` on load | Match save framework to load framework |
| Not specifying `batchable=True` | Batching disabled | Add `signatures={'predict': {'batchable': True}}` |
| Missing `bentofile.yaml` | Build fails | Create it with `bentoml build` from proper directory |

## Interview Questions and Answers

In [ ]:
qa = [
    {"q": "What is a Bento and what does it contain?",
     "a": """A Bento is a self-contained, deployable package for an ML service. It contains:

1. Model artifacts: weights, parameters from BentoML model store
2. Service code: Python files defining the API (service.py)
3. Python requirements: all pip packages needed
4. Docker base image: what to build the container from
5. API schema: OpenAPI spec generated from the service
6. Metadata: creation time, model versions, labels

It's like a Docker image but ML-aware: it knows which models it uses,
what versions, and how to serve them.

Build a Bento: bentoml build → creates a versioned artifact
Containerize: bentoml containerize <name>:<version> → Docker image

The key value: one command to go from model + code to a production Docker image."""},

    {"q": "BentoML vs FastAPI + Docker — what's the difference?",
     "a": """FastAPI + Docker (DIY approach):
- Write FastAPI service yourself
- Write Dockerfile yourself
- Handle model loading, versioning, batching manually
- More control, more work
- Good if you have specific requirements or already know FastAPI well

BentoML:
- Model store: centralized versioning for your models
- Runner abstraction: adaptive batching, GPU scheduling built-in
- One command containerization: no Dockerfile needed
- Deploy to BentoCloud, AWS SageMaker, GCP Vertex, etc. with integrations
- Standardized across your organization

Choose FastAPI for: existing FastAPI codebase, custom middleware needs,
                    when you want full control
Choose BentoML for: standardized ML serving, adaptive batching needed,
                    multiple models to manage, cloud deployment"""},

    {"q": "What is adaptive batching and why does it matter?",
     "a": """Adaptive batching automatically groups concurrent requests into a single model call.

Why it matters for ML models:
- Modern ML models (especially on GPU) are faster per-sample in batches
- GPU: 1 sample vs 64 samples → similar wall-clock time (parallelism)
- Without batching: 64 concurrent users → 64 sequential model.predict() calls
- With adaptive batching: 64 concurrent users → 1 model.predict(batch_64) call

How it works:
1. Request arrives, added to pending queue
2. Wait up to max_latency_ms for more requests
3. When max_batch_size reached OR timeout expires → process batch
4. Split results and return to individual requests

Typical speedup: 5-50× throughput improvement for GPU models
Cost: slight latency increase (max_latency_ms) to collect batches
Trade-off: tune max_latency_ms for your latency/throughput requirements"""},
]

for i, item in enumerate(qa, 1):
    print(f"Q{i}: {item['q']}")
    print(f"A:  {item['a'].strip()}")
    print("-" * 65)
    print()

## Summary

| Step | BentoML API |
|------|------------|
| Save model | `bentoml.sklearn.save_model(name, model, signatures=..., metadata=...)` |
| Load model | `bentoml.sklearn.load_model('name:version')` |
| Create runner | `bentoml.sklearn.get('name:latest').to_runner()` |
| Create service | `bentoml.Service(name=..., runners=[runner])` |
| Define API | `@svc.api(input=..., output=...)` |
| Serve locally | `bentoml serve service:svc` |
| Build Bento | `bentoml build` |
| Containerize | `bentoml containerize name:version` |

### Next Steps
1. **BentoML quickstart**: [https://docs.bentoml.com/en/latest/get-started/quickstart.html](https://docs.bentoml.com/en/latest/get-started/quickstart.html)
2. **BentoML cloud deploy**: [https://bentocloud.bentoml.com/](https://bentocloud.bentoml.com/)
3. **Next**: Ray Serve for distributed, auto-scaling serving across a cluster